# Dataset Description – Pakistani Rs.  5000 Banknote Classification

**Project:** Pakistani Rs. 5000 Banknote Verification  
**Classification model:** MobileNetV3-Large  
**Classes:** `genuine` and `counterfeit`  
**Purpose:** Document the dataset, class distribution, preprocessing, quality checks, and limitations.

## 1. Project and dataset overview

This dataset is intended for an experimental two-class image-classification task involving photographs of Pakistani Rs. 5000 banknotes. Each image is assigned one label:

- **Genuine:** an image labelled as a genuine banknote in the collected dataset.
- **Counterfeit:** an image labelled as counterfeit or as a simulated/altered example for this project.

The model learns visual patterns from the supplied images and predicts one of these labels. Its output is an experimental screening prediction, not an official determination of banknote authenticity.

### Dataset limitation

The counterfeit class includes user-created altered examples such as blur, added noise, removed or changed details, and heavy pixelation. These examples may not represent the full range of real-world counterfeit banknotes. Strong evaluation scores on this dataset therefore do not establish reliable performance on all circulating banknotes.


## 2. Dataset source and organization

The dataset was exported from the Roboflow project **`pkr-5000-verification`**, workspace **`rajas-workspace-fhlm5`**, version **5**, in folder format. The exported folder was previously identified as `PKR-5000-Verification-5`.

The Google Drive project structure is:

```text
PKR_5000_FYP/
├── Dataset/
├── Models/
└── Reports/
```

classification split structure:

```text
Dataset/
└── PKR-5000-Verification-5/
    ├── train/
    │   ├── genuine/
    │   └── counterfeit/
    ├── valid/  (or validation/)
    │   ├── genuine/
    │   └── counterfeit/
    └── test/
        ├── genuine/
        └── counterfeit/
```

The notebook searches for the split folders rather than assuming one exact exported folder name.


In [ ]:
# 3. Mount Google Drive and locate the dataset
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/PKR_5000_FYP')
DATASET_DIR = PROJECT_DIR / 'Dataset'
REPORT_DIR = PROJECT_DIR / 'Reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

assert DATASET_DIR.exists(), f'Dataset folder not found: {DATASET_DIR}'

def find_dataset_root(base_dir):
    valid_names = {'valid', 'validation'}
    for current, dirs, files in os.walk(base_dir):
        names = set(dirs)
        if 'train' in names and 'test' in names and names.intersection(valid_names):
            return Path(current)
    raise FileNotFoundError(
        'Could not find a folder containing train, valid/validation, and test. '
        'Check the Dataset folder structure.'
    )

DATASET_ROOT = find_dataset_root(DATASET_DIR)
VALID_DIR_NAME = 'valid' if (DATASET_ROOT / 'valid').exists() else 'validation'

print('Dataset root:', DATASET_ROOT)
print('Training folder:', DATASET_ROOT / 'train')
print('Validation folder:', DATASET_ROOT / VALID_DIR_NAME)
print('Test folder:', DATASET_ROOT / 'test')


## 4. Previously recorded dataset distribution

The following counts were recorded earlier in the project. The next cell checks the files currently present in Google Drive. If the counts differ, use the newly calculated counts as the authoritative description of the current dataset copy.

| Split | Genuine | Counterfeit | Total |
|---|---:|---:|---:|
| Training | 888 | 716 | 1,604 |
| Validation | 124 | 105 | 229 |
| Test | 61 | 53 | 114 |
| **Total** | **1,073** | **874** | **1,947** |

These figures must be verified against the actual folders before being presented as final project statistics.


In [ ]:
# 5. Count images in each split and class
from collections import defaultdict
import pandas as pd

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
CLASS_NAMES = ['genuine', 'counterfeit']
SPLITS = {
    'train': DATASET_ROOT / 'train',
    'validation': DATASET_ROOT / VALID_DIR_NAME,
    'test': DATASET_ROOT / 'test'
}

records = []
for split_name, split_path in SPLITS.items():
    for class_name in CLASS_NAMES:
        class_dir = split_path / class_name
        if not class_dir.exists():
            print(f'WARNING: Missing class folder: {class_dir}')
            count = 0
        else:
            count = sum(
                1 for p in class_dir.rglob('*')
                if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
            )
        records.append({'Split': split_name, 'Class': class_name, 'Images': count})

distribution_df = pd.DataFrame(records)
display(distribution_df.pivot(index='Split', columns='Class', values='Images').fillna(0).astype(int))

print('Totals by split:')
display(distribution_df.groupby('Split')['Images'].sum().to_frame('Total images'))

print('Totals by class:')
display(distribution_df.groupby('Class')['Images'].sum().to_frame('Total images'))

print('Overall image count:', int(distribution_df['Images'].sum()))
distribution_df.to_csv(REPORT_DIR / 'dataset_distribution.csv', index=False)
print('Saved:', REPORT_DIR / 'dataset_distribution.csv')


## 6. Class balance

Class balance describes how image counts are distributed between the two labels. A difference in class counts is not automatically a problem, but it should be reported because imbalance can affect model training and evaluation. Accuracy should be considered together with precision, recall, F1-score, and a confusion matrix.


In [ ]:
# 6.1 Visualize the number of images per class in each split
import matplotlib.pyplot as plt

pivot_df = distribution_df.pivot(
    index='Split', columns='Class', values='Images'
).fillna(0)

ax = pivot_df.reindex(['train', 'validation', 'test']).plot(
    kind='bar', figsize=(9, 5), rot=0
)
ax.set_title('Dataset Class Distribution by Split')
ax.set_xlabel('Dataset split')
ax.set_ylabel('Number of images')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'dataset_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()


## 7. Sample image inspection

Images may differ in resolution, aspect ratio, lighting, background, orientation, and camera quality. Inspecting representative samples helps identify unusual files and understand the visual variation in the dataset.


In [ ]:
# 7.1 Display sample images from the training split
from PIL import Image
import random

random.seed(42)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for row, class_name in enumerate(CLASS_NAMES):
    class_dir = SPLITS['train'] / class_name
    image_paths = [
        p for p in class_dir.rglob('*')
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]
    chosen = random.sample(image_paths, min(4, len(image_paths))) if image_paths else []

    for col, ax in enumerate(axes[row]):
        ax.axis('off')
        if col < len(chosen):
            try:
                with Image.open(chosen[col]) as im:
                    ax.imshow(im.convert('RGB'))
                ax.set_title(class_name)
            except Exception as e:
                ax.set_title(f'Could not open image: {e}')

plt.suptitle('Example Training Images')
plt.tight_layout()
plt.show()


## 8. Image dimensions and file integrity

The MobileNetV3-Large classification pipeline resizes input images to **224 × 224 pixels**. Original images may have different dimensions, but they should be readable and contain useful visual information. The next cell records original dimensions and identifies files that cannot be opened.


In [ ]:
# 8.1 Inspect image dimensions and unreadable files
dimension_records = []
unreadable_files = []

for split_name, split_path in SPLITS.items():
    for class_name in CLASS_NAMES:
        class_dir = split_path / class_name
        if not class_dir.exists():
            continue
        for path in class_dir.rglob('*'):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                try:
                    with Image.open(path) as im:
                        width, height = im.size
                        dimension_records.append({
                            'Split': split_name,
                            'Class': class_name,
                            'File': str(path),
                            'Width': width,
                            'Height': height,
                            'Format': im.format
                        })
                except Exception as e:
                    unreadable_files.append({'File': str(path), 'Error': str(e)})

dimensions_df = pd.DataFrame(dimension_records)
if not dimensions_df.empty:
    display(dimensions_df[['Width', 'Height']].describe())
    print('Most common dimensions:')
    display(
        dimensions_df.groupby(['Width', 'Height'])
        .size().sort_values(ascending=False).head(10).to_frame('Count')
    )

print('Readable images:', len(dimensions_df))
print('Unreadable images:', len(unreadable_files))
if unreadable_files:
    display(pd.DataFrame(unreadable_files).head(20))

dimensions_df.to_csv(REPORT_DIR / 'image_dimensions.csv', index=False)
print('Saved image dimensions:', REPORT_DIR / 'image_dimensions.csv')


## 9. Preprocessing for MobileNetV3-Large

The classification workflow uses the following input preparation:

1. Load the image and convert it to RGB.
2. Resize it to 224 × 224 pixels.
3. Convert it to a tensor.
4. Normalize the channels using ImageNet mean and standard deviation when using ImageNet-pretrained MobileNetV3-Large weights.

ImageNet normalization values:

- Mean: `[0.485, 0.456, 0.406]`
- Standard deviation: `[0.229, 0.224, 0.225]`

The exact transformations used for final evaluation must match those used during model training.


In [ ]:
# 9.1 Example preprocessing pipeline (PyTorch)
from torchvision import transforms

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print(image_transform)


## 10. Dataset splitting and leakage prevention

The dataset is organized into three subsets:

- **Training set:** used to update model parameters.
- **Validation set:** used during development to monitor performance and make model-selection decisions.
- **Test set:** should remain untouched until final evaluation to provide an independent estimate of performance.

### Augmented-image leakage

If multiple augmented images were created from the same original photograph, related images should remain in the same split and, during grouped cross-validation, in the same fold. Otherwise, very similar images can appear in both training and validation data, making evaluation scores overly optimistic. A filename-based grouping rule must be checked against the actual exported filenames rather than assumed to be correct.

The test set must not be used to tune the model, choose epochs, or repeatedly adjust thresholds.


## 11. Dataset limitations and considerations

The following limitations should be acknowledged:

1. **Limited sample size:** the dataset is relatively small compared with the diversity of banknotes encountered in real circulation.
2. **Simulated counterfeit examples:** digitally altered images may not capture the materials, printing methods, or security-feature characteristics of actual counterfeit notes.
3. **Image conditions:** lighting, blur, camera angle, focus, resolution, and background can affect classification.
4. **Possible source bias:** if genuine and counterfeit images were captured under noticeably different conditions, the model may learn background or capture-related cues rather than banknote characteristics.
5. **Label quality:** class labels and image quality should be reviewed; incorrect labels can affect training and evaluation.
6. **No guarantee of authenticity:** the model is an experimental image-screening tool and should not be treated as an official currency-authentication system.

For stronger future evaluation, use a larger and more diverse dataset with independently verified labels, multiple capture devices and environments, and an untouched test set that reflects realistic use conditions.


## 12. Conclusion

The dataset provides a two-class image collection for developing and evaluating a MobileNetV3-Large classifier for Pakistani Rs. 5000 banknote images. It contains images labelled as genuine and counterfeit and is arranged into training, validation, and test subsets.

The previously recorded dataset contained **1,947 images: 1,073 genuine and 874 counterfeit**. These totals must be checked against the current dataset files. Evaluation results should be interpreted in light of the dataset's size, simulated counterfeit examples, possible capture-condition bias, and the need to prevent augmented-image leakage.

This notebook is both a written dataset description and a reproducible inspection workflow. Run the code cells in order in Google Colab to refresh the counts, charts, and inspection files.
